In [0]:
from pyspark.sql.functions import col, to_timestamp, lit, current_timestamp, coalesce, when
from delta.tables import DeltaTable

#lectura tablas de bronze
df_sensors_bronze = spark.read.table("agro_catalog.bronce.sensor_readings_bronze")
df_weather_bronze = spark.read.table("agro_catalog.bronce.weather_readings_bronze")
df_irrig_bronze = spark.read.table("agro_catalog.bronce.irrigation_events_bronze")
df_alerts_bronze = spark.read.table("agro_catalog.bronce.alerts_bronze")

def guardar_incremental_merge(df_nuevos, nombre_tabla_destino, clave_primaria="event_id"):
    """Realiza un merge into si la tabla existe sino se crea"""
    if spark.catalog.tableExists(nombre_tabla_destino):
        tabla_delta = DeltaTable.forName(spark, nombre_tabla_destino)
        (tabla_delta.alias("destino")
            .merge(
                df_nuevos.alias("nuevos"),
                f"destino.{clave_primaria} = nuevos.{clave_primaria}"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
        print(f"Carga incremental (MERGE) completada en: {nombre_tabla_destino}")
    else:
        df_nuevos.write.format("delta").saveAsTable(nombre_tabla_destino)
        print(f"Tabla creada por primera vez: {nombre_tabla_destino}")



#SENSORES DE SUELO 

print("Procesando sensores de suelo...")
#desanidado y normalizado
df_sensors_plano = df_sensors_bronze.select(
    col("event_id"),
    to_timestamp(col("timestamp"), "yyyy-MM-dd'T'HH:mm:ss'Z'").alias("timestamp"),
    col("device_id"),
    col("plot_id"),
    col("crop"),
    col("sensor_depth_cm"),
    col("battery_pct"),
    col("device_status"),
    col("measurements.soil_moisture_pct").alias("soil_moisture_pct"),
    col("measurements.soil_temperature_c").alias("soil_temperature_c"),
    col("measurements.soil_ph").alias("soil_ph"),
    col("measurements.soil_ec_ds_m").alias("soil_ec_ds_m")
).dropDuplicates(["event_id"])
# regals de calidad como no nullos y rangos validos
condicion_calidad_sensores = (
    col("event_id").isNotNull() &
    col("timestamp").isNotNull() &
    col("plot_id").isNotNull() &
    col("battery_pct").isNotNull() &
    col("soil_moisture_pct").isNotNull() &
    (col("battery_pct").between(0, 100)) &
    (col("soil_moisture_pct").between(0, 100))
)

df_sensors_validos = df_sensors_plano.filter(condicion_calidad_sensores)

motivo_sensor = (
    when(col("soil_moisture_pct").isNull() | col("battery_pct").isNull(), lit("Ausencia de lectura en sensor (Valores Nulos)"))
    .otherwise(lit("Valor fuera de rango fisico [0-100] (bateria/humedad)"))
)

df_sensors_rechazados = (
    df_sensors_plano.filter(~coalesce(condicion_calidad_sensores, lit(False)))
    .select(
        col("event_id"),
        col("timestamp"),
        col("device_id"),
        lit("sensor_readings").alias("origen"),
        motivo_sensor.alias("motivo_rechazo")
    )
)

guardar_incremental_merge(df_sensors_validos, "agro_catalog.bronce.sensor_readings_silver")


# CLIMA

print("Procesando datos de clima...")

df_weather_plano = df_weather_bronze.select(
    col("event_id"),
    to_timestamp(col("timestamp"), "yyyy-MM-dd'T'HH:mm:ss'Z'").alias("timestamp"),
    col("device_id"),
    col("battery_pct"),
    col("device_status"),
    col("measurements.air_temperature_c").alias("air_temperature_c"),
    col("measurements.air_humidity_pct").alias("air_humidity_pct"),
    col("measurements.solar_radiation_w_m2").alias("solar_radiation_w_m2"),
    col("measurements.rainfall_mm").alias("rainfall_mm"),
    col("measurements.wind_speed_ms").alias("wind_speed_ms")
).dropDuplicates(["event_id"])

condicion_calidad_clima = (
    col("event_id").isNotNull() &
    col("timestamp").isNotNull() &
    col("battery_pct").isNotNull() &
    col("rainfall_mm").isNotNull() &
    (col("battery_pct").between(0, 100)) &
    (col("rainfall_mm") >= 0)
)

df_weather_validos = df_weather_plano.filter(condicion_calidad_clima)
df_weather_rechazados = (
    df_weather_plano.filter(~coalesce(condicion_calidad_clima, lit(False)))
    .select(
        col("event_id"),
        col("timestamp"),
        col("device_id"),
        lit("weather_readings").alias("origen"),
        lit("Nulos o valores negativos en clima").alias("motivo_rechazo")
    )
)

guardar_incremental_merge(df_weather_validos, "agro_catalog.bronce.weather_readings_silver")


# riegos
print("Procesando eventos de riego...")

df_irrig_plano = df_irrig_bronze.select(
    col("event_id"),
    to_timestamp(col("timestamp"), "yyyy-MM-dd'T'HH:mm:ss'Z'").alias("timestamp"),
    col("device_id"),
    col("plot_id"),
    col("irrigation_status"),
    col("valve_status"),
    col("water_flow_l_min"),
    col("water_pressure_bar"),
    col("water_volume_l"),
    col("reason"),
    col("battery_pct"),
    col("device_status")
).dropDuplicates(["event_id"])

condicion_calidad_riego = (
    col("event_id").isNotNull() &
    col("timestamp").isNotNull() &
    col("water_volume_l").isNotNull() &
    col("water_flow_l_min").isNotNull() &
    (col("water_volume_l") >= 0) &
    (col("water_flow_l_min") >= 0)
)

df_irrig_validos = df_irrig_plano.filter(condicion_calidad_riego)
df_irrig_rechazados = (
    df_irrig_plano.filter(~coalesce(condicion_calidad_riego, lit(False)))
    .select(
        col("event_id"),
        col("timestamp"),
        col("device_id"),
        lit("irrigation_events").alias("origen"),
        lit("Volumen o caudal negativo/nulo").alias("motivo_rechazo")
    )
)

guardar_incremental_merge(df_irrig_validos, "agro_catalog.bronce.irrigation_events_silver")



# Alertas
print("Procesando alertas operativas...")

df_alerts_plano = df_alerts_bronze.select(
    col("event_id"),
    to_timestamp(col("timestamp"), "yyyy-MM-dd'T'HH:mm:ss'Z'").alias("timestamp"),
    col("device_id"),
    coalesce(col("plot_id"), lit("GENERAL_GRANJA")).alias("plot_id"),
    col("farm_id"),
    col("event_type"),
    col("alert_code"),
    col("severity"),
    col("measured_value"),
    col("threshold_value"),
    col("message")
).dropDuplicates(["event_id"])

condicion_calidad_alertas = (
    col("event_id").isNotNull() &
    col("timestamp").isNotNull() &
    col("alert_code").isNotNull()
)

df_alerts_validos = df_alerts_plano.filter(condicion_calidad_alertas)
df_alerts_rechazados = (
    df_alerts_plano.filter(~coalesce(condicion_calidad_alertas, lit(False)))
    .select(
        col("event_id"),
        col("timestamp"),
        col("device_id"),
        lit("alerts").alias("origen"),
        lit("Alerta sin identificador o sin codigo").alias("motivo_rechazo")
    )
)

guardar_incremental_merge(df_alerts_validos, "agro_catalog.bronce.alerts_silver")


# rechazos 
df_todos_rechazos = (
    df_sensors_rechazados
    .unionByName(df_weather_rechazados)
    .unionByName(df_irrig_rechazados)
    .unionByName(df_alerts_rechazados)
    .withColumn("fecha_auditoria", current_timestamp())
)

df_todos_rechazos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("agro_catalog.bronce.calidad_rechazos_silver")

print("\n--- RESUMEN DE VALIDACION Y CALIDAD (ORIGEN VS RESULTADO) ---")
print(f"Sensores  -> Bronze: {df_sensors_bronze.count()} | Sin duplicados: {df_sensors_plano.count()} | Silver Validos: {df_sensors_validos.count()} | Rechazados: {df_sensors_rechazados.count()}")
print(f"Clima     -> Bronze: {df_weather_bronze.count()} | Sin duplicados: {df_weather_plano.count()} | Silver Validos: {df_weather_validos.count()} | Rechazados: {df_weather_rechazados.count()}")
print(f"Riego     -> Bronze: {df_irrig_bronze.count()} | Sin duplicados: {df_irrig_plano.count()} | Silver Validos: {df_irrig_validos.count()} | Rechazados: {df_irrig_rechazados.count()}")
print(f"Alertas   -> Bronze: {df_alerts_bronze.count()} | Sin duplicados: {df_alerts_plano.count()} | Silver Validos: {df_alerts_validos.count()} | Rechazados: {df_alerts_rechazados.count()}")
print(f"Total registros en tabla calidad_rechazos_silver: {df_todos_rechazos.count()}")

Procesando sensores de suelo...
Carga incremental (MERGE) completada en: agro_catalog.bronce.sensor_readings_silver
Procesando datos de clima...
Carga incremental (MERGE) completada en: agro_catalog.bronce.weather_readings_silver
Procesando eventos de riego...
Carga incremental (MERGE) completada en: agro_catalog.bronce.irrigation_events_silver
Procesando alertas operativas...
Carga incremental (MERGE) completada en: agro_catalog.bronce.alerts_silver

--- RESUMEN DE VALIDACION Y CALIDAD (ORIGEN VS RESULTADO) ---
Sensores  -> Bronze: 6190 | Sin duplicados: 6190 | Silver Validos: 6154 | Rechazados: 36
Clima     -> Bronze: 619 | Sin duplicados: 619 | Silver Validos: 619 | Rechazados: 0
Riego     -> Bronze: 3095 | Sin duplicados: 3095 | Silver Validos: 3095 | Rechazados: 0
Alertas   -> Bronze: 98 | Sin duplicados: 98 | Silver Validos: 98 | Rechazados: 0
Total registros en tabla calidad_rechazos_silver: 36


Segunda carga 

In [0]:
from pyspark.sql.functions import lit

# Registro existente con una actualizacion
df_modificado = (
    spark.read.table("agro_catalog.bronce.sensor_readings_silver")
    .limit(1)
    .withColumn("battery_pct", lit(99.9))
)

#Registro nuevo
df_nuevo = (
    spark.read.table("agro_catalog.bronce.sensor_readings_silver")
    .limit(1)
    .withColumn("event_id", lit("evt_prueba_incremental_001"))
)

df_lote_2 = df_modificado.unionByName(df_nuevo)

# Ejecucion del merge
guardar_incremental_merge(df_lote_2, "agro_catalog.bronce.sensor_readings_silver")

Carga incremental (MERGE) completada en: agro_catalog.bronce.sensor_readings_silver


In [0]:
%sql
DESCRIBE HISTORY agro_catalog.bronce.sensor_readings_silver;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
11,2026-11-19T21:05:35.000Z,76537812757437,sugussanchez31@gmail.com,MERGE,"Map(predicate -> [""(event_id#17264 = event_id#17288)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}], clusteringOnWriteStatus -> null)",null,List(2334728865274307),bcc283c0-b2ca-41d7-b6f8-a0b8848b6d0e,1119-205152-8eljb1wp-v2n,10,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 8316, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 4559, materializeSourceTimeMs -> 823, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1026, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2662)",null,Databricks-Runtime/19.15.x-aarch64-photon-scala2.13
10,2026-11-19T21:04:52.000Z,76537812757437,sugussanchez31@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2334728865274307),18086556-94d8-4a99-b31f-6ea6f80aa61a,1119-205152-8eljb1wp-v2n,9,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 339710, p25FileSize -> 170472, numDeletionVectorsRemoved -> 1, minFileSize -> 170472, numAddedFiles -> 1, maxFileSize -> 170472, p75FileSize -> 170472, p50FileSize -> 170472, numAddedBytes -> 170472)",null,Databricks-Runtime/19.15.x-aarch64-photon-scala2.13
9,2026-11-19T21:04:48.000Z,76537812757437,sugussanchez31@gmail.com,MERGE,"Map(predicate -> [""(event_id#10322 = event_id#10356)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}], clusteringOnWriteStatus -> null)",null,List(2334728865274307),18086556-94d8-4a99-b31f-6ea6f80aa61a,1119-205152-8eljb1wp-v2n,8,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 169298, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 6154, executionTimeMs -> 14240, materializeSourceTimeMs -> 3970, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 5791, numTargetRowsUpdated -> 6154, numOutputRows -> 6154, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 6154, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 4247)",null,Databricks-Runtime/19.15.x-aarch64-photon-scala2.13
8,2026-11-14T23:05:46.000Z,76537812757437,sugussanchez31@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2334728865274307),4e1fcb3e-8985-4920-90f0-765d7dd3f062,1114-222125-rz5mjrsy-v2n,7,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 339709, p25FileSize -> 170412, numDeletionVectorsRemoved -> 1, minFileSize -> 170412, numAddedFiles -> 1, maxFileSize -> 170412, p75FileSize -> 170412, p50FileSize -> 170412, numAddedBytes -> 170412)",null,Databricks-Runtime/19.14.x-aarch64-photon-scala2.13
7,2026-11-14T23:05:43.000Z,76537812757437,sugussanchez31@gmail.com,MERGE,"Map(predicate -> [""(event_id#36749 = event_id#36783)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates 

mostrado de los datos anteriores de manera lejible 

In [0]:
display(
    spark.sql("DESCRIBE HISTORY agro_catalog.bronce.sensor_readings_silver")
    .select(
        "version",
        "timestamp",
        "operation",
        "operationMetrics.numSourceRows",
        "operationMetrics.numTargetRowsInserted",
        "operationMetrics.numTargetRowsUpdated"
    )
    .limit(4)
)

version,timestamp,operation,numSourceRows,numTargetRowsInserted,numTargetRowsUpdated
11,2026-11-19T21:05:35.000Z,MERGE,2,1,1
10,2026-11-19T21:04:52.000Z,OPTIMIZE,null,null,null
9,2026-11-19T21:04:48.000Z,MERGE,6154,0,6154
8,2026-11-14T23:05:46.000Z,OPTIMIZE,null,null,null
